In [1]:
import os
import json
import faiss
import heapq
import numpy as np
from transformers import AutoProcessor, AutoModel
import torch
import pickle
from pathlib import Path
from typing import List, Optional
from tqdm import tqdm

# ============================================================
# UTILITY: Chunk list
# ============================================================

def chunk_list(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i + n]


# ============================================================
# LOAD PROCESSED PROMPTS FROM JSONL
# ============================================================

def load_processed_prompts_from_jsonl(jsonl_path: str):
    processed = set()
    path = Path(jsonl_path)

    if not path.exists():
        return processed

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                p = obj.get("prompt")
                if p:
                    processed.add(p)
            except json.JSONDecodeError:
                continue

    print(f"[Resume] Found {len(processed)} previously processed prompts.")
    return processed


# ============================================================
# APPEND RESULTS TO JSONL
# ============================================================

def append_results_to_jsonl(jsonl_path: str, batch_results: list):
    path = Path(jsonl_path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with open(path, "a", encoding="utf-8") as f:
        for entry in batch_results:
            f.write(json.dumps(entry) + "\n")
            f.flush()
            os.fsync(f.fileno())

    print(f"[JSONL Append] {len(batch_results)} prompts appended → {jsonl_path}")


# ============================================================
# FAISS SEARCH FUNCTION (GENERIC POSITIVE + NEGATIVE FILTERING)
# ============================================================

def search_images(
    shard_index_paths: List[str],
    shard_mapping_paths: List[str],
    model_name: str,
    prompts: List[str],
    generic_positive_prompts: List[str],  # <<< NEW: e.g., ["person", "human face"]
    negative_prompts: Optional[List[str]] = None,
    top_k: int = 10,
    similarity_threshold: Optional[float] = None,
    dominance_margin: float = 0.05,  # Increased default
    device: str = None
):
    """
    Batch FAISS search across shards with TWO-STAGE filtering:
    
    1. Search using specific prompts (e.g., "Male Accountant")
    2. Filter results by comparing image to:
       - generic_positive_prompts: ["person", "human face", "portrait"]
       - negative_prompts: ["document", "paper", "logo", ...]
    
    Rejects an image if:
        max(sim(image, negative)) >= max(sim(image, generic_positive)) - margin
    """

    device = device or ("cuda" if torch.cuda.is_available() else "cpu")

    # -------------------- Load CLIP --------------------
    processor = AutoProcessor.from_pretrained(model_name, force_download=False)
    model = AutoModel.from_pretrained(model_name, force_download=False).to(device)
    model.eval()

    # -------------------- Encode SEARCH prompts (specific) --------------------
    pos_inputs = processor(
        text=prompts,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(device)

    with torch.no_grad():
        pos_text_features = model.get_text_features(**pos_inputs)

    pos_text_features = pos_text_features / pos_text_features.norm(dim=-1, keepdim=True)
    pos_query_embs = pos_text_features.cpu().numpy().astype("float32")
    faiss.normalize_L2(pos_query_embs)

    n_queries = pos_query_embs.shape[0]
    all_results = [[] for _ in range(n_queries)]

    # -------------------- Encode GENERIC POSITIVE prompts (for filtering) --------------------
    generic_pos_inputs = processor(
        text=generic_positive_prompts,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(device)

    with torch.no_grad():
        generic_pos_features = model.get_text_features(**generic_pos_inputs)

    generic_pos_features = generic_pos_features / generic_pos_features.norm(dim=-1, keepdim=True)
    generic_pos_embs = generic_pos_features.cpu().numpy().astype("float32")

    # -------------------- Encode NEGATIVE prompts (for filtering) --------------------
    neg_embs = None
    if negative_prompts:
        neg_inputs = processor(
            text=negative_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True
        ).to(device)

        with torch.no_grad():
            neg_features = model.get_text_features(**neg_inputs)

        neg_features = neg_features / neg_features.norm(dim=-1, keepdim=True)
        neg_embs = neg_features.cpu().numpy().astype("float32")

    # -------------------- Search shards --------------------
    total_retrieved = 0
    total_filtered = 0
    
    for idx_path, map_path in tqdm(
        list(zip(shard_index_paths, shard_mapping_paths)),
        desc="Searching shards",
        unit="shard"
    ):
        idx_path = Path(idx_path)
        map_path = Path(map_path)

        shard_name = idx_path.stem
        parts = shard_name.split("_")
        group_id = parts[1] if len(parts) > 1 else "unknown"

        index = faiss.read_index(str(idx_path))
        with open(map_path, "rb") as f:
            idx_to_path = pickle.load(f)

        per_shard_k = min(len(idx_to_path), top_k) if top_k != None else len(idx_to_path)
        D, I = index.search(pos_query_embs, per_shard_k)

        for qi in range(n_queries):
            pos_query_emb = pos_query_embs[qi]

            for pos_sim, idx in zip(D[qi], I[qi]):
                idx = int(idx)
                if idx < 0 or idx >= len(idx_to_path):
                    continue
                if similarity_threshold is not None and pos_sim < similarity_threshold:
                    continue

                total_retrieved += 1

                # Retrieve exact image embedding
                img_emb = index.reconstruct(idx)
                img_emb = img_emb / np.linalg.norm(img_emb)

                # ---------------- GENERIC POSITIVE vs NEGATIVE FILTERING ----------------
                # Compare image to GENERIC "person/face" concepts
                generic_pos_sim = np.max(img_emb @ generic_pos_embs.T)
                
                if neg_embs is not None:
                    # Compare image to negative concepts
                    neg_sim = np.max(img_emb @ neg_embs.T)

                    # Reject if negative dominates generic positive
                    if neg_sim >= (generic_pos_sim - dominance_margin):
                        total_filtered += 1
                        continue

                all_results[qi].append({
                    "image_path": idx_to_path[idx],
                    "score": float(pos_sim),  # Original search score
                    "generic_person_score": float(generic_pos_sim),  # Debug info
                    "shard": shard_name,
                    "group_id": group_id
                })

        del index

    print(f"[Filtering Stats] Retrieved: {total_retrieved:,} | Filtered out: {total_filtered:,} | Kept: {total_retrieved - total_filtered:,}")

    # -------------------- Global Top-K --------------------
    final_results = []
    if top_k != None:
        for qi in range(n_queries):
            final_results.append({
                "prompt": prompts[qi],
                "results": heapq.nlargest(
                    top_k,
                    all_results[qi],
                    key=lambda x: x["score"]
                )
            })
    else:
        for qi in range(n_queries):
            final_results.append({
                "prompt": prompts[qi],
                "results": sorted(
                    all_results[qi],
                    key=lambda x: x["score"],
                    reverse=True
                )
            })

    return final_results


# ============================================================
# MAIN PROCESSING LOOP (WITH RESUME)
# ============================================================

def process_all_prompts_with_resume(
    shard_indexes,
    shard_mappings,
    profession_list,
    prompt_templates,
    jsonl_path,
    chunk_size,
    generic_positive_prompts=[
        "a photo of a person",
        "human face",
        "person portrait",
        "people"
    ],
    negative_prompts=[
        "document",
        "paper document",
        "text on paper",
        "printed form",
        "business card",
        "logo",
        "company logo",
        "signboard",
        "clothing item",
        "empty shirt",
        "suit on hanger",
        "office supplies",
        "calculator",
        "keyboard",
        "computer screen"
    ],
    model_name="openai/clip-vit-large-patch14",
    similarity_threshold=0.15,
    dominance_margin=0.05,
    top_k=1000
):
    print("UPDATED CODE - Generic Positive Filtering")
    processed = load_processed_prompts_from_jsonl(jsonl_path)

    all_prompts = []
    for p in sorted(set(profession_list)):
        for tmpl in prompt_templates:
            all_prompts.append(tmpl.format(object=p))

    for bidx, chunk in enumerate(chunk_list(all_prompts, chunk_size)):
        to_process = [p for p in chunk if p not in processed]

        if not to_process:
            print(f"[Batch {bidx}] All prompts already processed → Skipping.")
            continue

        print(f"[Batch {bidx}] Processing {len(to_process)} prompts: {to_process}")

        batch_results = search_images(
            shard_index_paths=shard_indexes,
            shard_mapping_paths=shard_mappings,
            model_name=model_name,
            prompts=to_process,
            generic_positive_prompts=generic_positive_prompts,
            negative_prompts=negative_prompts,
            top_k=top_k,
            similarity_threshold=similarity_threshold,
            dominance_margin=dominance_margin
        )

        append_results_to_jsonl(jsonl_path, batch_results)

        for r in batch_results:
            processed.add(r["prompt"])

    print("All batches processed.")

c:\Users\User\anaconda3\envs\opencv_cuda\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from pathlib import Path
from tqdm import tqdm

faiss_dir = r"G:\Thesis\image_retrieval_faiss_indices"

shard_indexes = sorted([str(p) for p in Path(faiss_dir).glob("faiss_*_IndexFlatIP.index")])
shard_mappings = sorted([str(p) for p in Path(faiss_dir).glob("faiss_*_mapping.pkl")])
assert len(shard_indexes) == len(shard_mappings), "Mismatch between indexes and mappings!"

prompt_templates = ["Male {object}", "Female {object}"]

# List of professions retrieved from papers as well as ChatGPT
profession_list = ["Accountant"]

prompt_templates = ["Male {object}", "Female {object}"]

jsonl_path = r"G:\Thesis\ImageRetrieval\test\retrieval_results_batchsize_10.jsonl"

process_all_prompts_with_resume(
    shard_indexes=shard_indexes,
    shard_mappings=shard_mappings,
    profession_list=profession_list,
    prompt_templates=prompt_templates,
    jsonl_path=jsonl_path,
    chunk_size=10,
    generic_positive_prompts=["person", "human", "face", "a photo of a person", "human face", "person portrait", "people"],
    negative_prompts = ["Cartoon", "NSFW", "Sex", "Naked", "Clothing", "Object", "Sign", "Logo", "Document", "Paper", "Page"],
    model_name="openai/clip-vit-large-patch14",
    similarity_threshold=0.15,
    dominance_margin=0.01,
    top_k=None #200_000
)

UPDATED CODE - Generic Positive Filtering
[Batch 0] Processing 2 prompts: ['Male Accountant', 'Female Accountant']


c:\Users\User\anaconda3\envs\opencv_cuda\lib\site-packages\huggingface_hub\file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Searching shards: 100%|██████████| 36/36 [28:43<00:00, 47.88s/shard]


[Filtering Stats] Retrieved: 21,975,857 | Filtered out: 18,461,011 | Kept: 3,514,846
[JSONL Append] 2 prompts appended → G:\Thesis\ImageRetrieval\test\retrieval_results_batchsize_10.jsonl
All batches processed.
